# Classifier Generalization Comparison

This read-only notebook compares held-out binary performance, long-text performance, SemEval localization, and BBC safety diagnostics.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from notebooks.utils.classification_results_analysis_utils import document_metrics, load_classification_run
from notebooks.utils.classifier_generalization_comparison_utils import (
    held_out_binary_metrics, plot_dataset_heatmap, plot_f1_mae_frontier,
    plot_generalization_leaderboard, plot_parameter_effects, plot_safety_frontier,
    comparison_matrix, configuration_index, held_out_binary_metrics,
    plot_matrix_facets, rank_generalization_candidates, select_comparison_metrics,
)
from notebooks.utils.report_diagrams_utils import plot_output_root, save_png_figure

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PLOT_RUN_ID = os.environ.get("PLOT_RUN_ID", "plots-regeneration-2026-10-26")
PLOT_OUTPUT_ROOT = plot_output_root(DATA_ROOT, "compare_classifier_generalization", plot_run_id=PLOT_RUN_ID)
RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
MIN_TEXT_TOKENS = 200
TOP_TIER_MACRO_F1_GAP = 0.02
results = load_classification_run(DATA_ROOT, run_id=RUN_ID)
metrics = document_metrics(results)
pd.set_option("display.max_columns", 50)

## Evidence And Cohorts

BBC is HG-only and is excluded from the primary binary ranking.

In [ ]:
held_out = held_out_binary_metrics(metrics)
if held_out.empty:
    display(Markdown("**insufficient_binary_holdout:** this run has no true unseen binary evaluation dataset."))
else:
    display(held_out.groupby(["training_regime", "dataset"], dropna=False).size().rename("rows").reset_index())
cohorts = {"All texts": select_comparison_metrics(metrics, min_text_tokens=None), f">= {MIN_TEXT_TOKENS}": select_comparison_metrics(metrics, min_text_tokens=MIN_TEXT_TOKENS)}
rankings = {}
for cohort_name, cohort_metrics in cohorts.items():
    display(configuration_index(cohort_metrics, facet_columns=("model_family", "training_window", "evaluation_window")))
    mean_matrices = comparison_matrix(cohort_metrics, value="macro_f1", aggregation="mean")
    worst_matrices = comparison_matrix(cohort_metrics, value="macro_f1", aggregation="worst")
    mean_figure = plot_matrix_facets(mean_matrices, title=f"{cohort_name}: mean macro-F1", metric="mean_macro_f1", facet_columns=("model_family", "training_window", "evaluation_window"))
    save_png_figure(mean_figure, PLOT_OUTPUT_ROOT / f"mean_macro_f1__{cohort_name.lower().replace(' ', '_').replace('>=', 'at_least')}__{RUN_ID}.png")
    display(mean_figure)
    worst_figure = plot_matrix_facets(worst_matrices, title=f"{cohort_name}: worst macro-F1", metric="worst_macro_f1", facet_columns=("model_family", "training_window", "evaluation_window"))
    save_png_figure(worst_figure, PLOT_OUTPUT_ROOT / f"worst_macro_f1__{cohort_name.lower().replace(' ', '_').replace('>=', 'at_least')}__{RUN_ID}.png")
    display(worst_figure)

## Decision Figures

Each figure is rendered for both cohorts.

In [ ]:
display(Markdown("Rows are scorer LLMs and columns are feature sets. Each panel fixes model family, training window, and evaluation window. Blank cells mean that configuration is unavailable."))